# AION-U1 Pilot (Colab)

Laeuft mit GPU (T4) oder auf CPU; auf CPU dauert die Messung mit 24 Faellen etwa 1-2 Stunden.

Vorher: den Ordner `sft_pilot_v3` aus Downloads als Datei **`sft_pilot_v3.zip`** in diesen Colab-Ordner hochladen (links das Dateisymbol). Zelle 3 entpackt ihn. Ohne Upload startet Zelle 3 ein neues Training.

In [ ]:
# 1) Repo
import os, subprocess
if not os.path.isdir('aion-u1'):
    subprocess.run(['git','clone','-q','-b','feat/pilot-ergebnis-v2','https://github.com/Reznap87/aion-u1.git'], check=True)
%cd aion-u1
!pip install -q -e '.[train]'
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2) fp32 bleibt eingestellt: fp16 war die Ursache fuer eval_loss NaN
from pathlib import Path
text = Path('training/configs/sft_lora_open_v3.yaml').read_text(encoding='utf-8')
print('fp16 aktiv:', 'fp16: true' in text, '| fp32:', 'fp16: false' in text)

In [ ]:
# 3) Adapter bereitstellen: Zip entpacken (beliebige Verschachtelung) oder Training
import pathlib, subprocess, shutil
ziel = pathlib.Path('artifacts/sft/pilot_open_v3')
if not (ziel / 'final_model').is_dir():
    zipdatei = pathlib.Path('sft_pilot_v3.zip')
    if not zipdatei.is_file():
        zipdatei = pathlib.Path('../sft_pilot_v3.zip')
    if zipdatei.is_file():
        subprocess.run(['unzip','-q','-o',str(zipdatei),'-d','_upload'], check=True)
        treffer = list(pathlib.Path('_upload').glob('**/final_model/adapter_model.safetensors'))
        if treffer:
            quelle = treffer[0].parent.parent.parent   # .../sft/<lauf>
            ziel.mkdir(parents=True, exist_ok=True)
            for teil in quelle.iterdir():
                shutil.copytree(teil, ziel / teil.name, dirs_exist_ok=True) if teil.is_dir() else shutil.copy2(teil, ziel / teil.name)
            print('Adapter uebernommen aus', quelle)
        else:
            print('Zip ohne Adapter gefunden')
    else:
        print('Kein Zip — Training startet (auf CPU sehr langsam).')
        subprocess.run(['aion-u1-train-sft','--config','training/configs/sft_lora_open_v3.yaml'], check=True)
print('final_model existiert:', (ziel / 'final_model').is_dir())

In [ ]:
# 4) Baseline des Basismodells (gleiche Suite, ohne Adapter)
!aion-u1-evaluate-model --base-model Qwen/Qwen2.5-1.5B-Instruct \n  --suite evaluations/capability_suite/benchmark_hard_text_v1.yaml \n  --device auto --max-tokens 512 \n  --output reports/eval-baseline.json

In [ ]:
# 5) Messung des trainierten Adapters (harte Text-Suite)
!aion-u1-evaluate-model --base-model Qwen/Qwen2.5-1.5B-Instruct \n  --adapter artifacts/sft/pilot_open_v3/final_model \n  --suite evaluations/capability_suite/benchmark_hard_text_v1.yaml \n  --device auto --max-tokens 512 \n  --output reports/eval-pilot.json

In [ ]:
# 6) Ergebnis lesen
import json
for name in ('eval-baseline', 'eval-pilot'):
    try:
        d = json.load(open(f'reports/{name}.json', encoding='utf-8'))
        s = list(d['scores'].values())[0]
        print(f"{name:14} index={s.get('overall_score')} gemessen={s.get('measured_score')} coverage={s.get('coverage')}")
    except FileNotFoundError:
        print(name, 'fehlt')
print('Zielmarke GOAL.md: 0.847 (beste Lehrer, harte Suite)')

In [ ]:
# 7) Adapter sichern
import os
if os.path.isdir('artifacts/sft/pilot_open_v3/final_model'):
    !zip -q -r sft_pilot_v3_ergebnis.zip artifacts/sft/pilot_open_v3/final_model
    from google.colab import files
    files.download('sft_pilot_v3_ergebnis.zip')
else:
    print('kein Adapter vorhanden')